In [1]:
"""
m/z-to-m/z Spatial Pattern Matching Pipeline for Isotope Detection
===================================================================
Compares each m/z feature with all other m/z features within the same sample
to identify isotope patterns based on spatial similarity scores.

Based on Gene-to-m/z Spatial Pattern Matching Pipeline V1 (Analytic - Optimized)
Modified to:
- Compare m/z features within each sample (no cross-modal alignment)
- Remove 180-degree rotation (same coordinate space)
- Skip visualizations
- Output similarity scores to CSV

Stage 3: Build parent-children hierarchy from identified isotopes.
"""

import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from typing import Dict, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings('ignore')

# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_PIXEL_SIZE = 60  # μm

# Mass differences for isotope and adduct detection
MASS_DIFFS = {
    'Isotope (M+1)': 1.0033,
    'Isotope (M+2)': 2.0067,
    'Adduct (NH4)': 17.0265,
    'Adduct (Na)': 21.982,
    'Adduct (K)': 37.9555
}

# Tolerance for mass difference matching (in Da)
MASS_DIFF_TOLERANCE = 0.01

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

# Isotope identification thresholds
MIN_ANIMALS = 12
MIN_SCORE = 60

# Output directory
OUTPUT_DIR = './mz_isotope_results_1'

# Stage 3 configuration
HIERARCHY_PRECISION = 4
HIERARCHY_TOLERANCE = 0.01


# =============================================================================
# STAGE 1: SPATIAL PATTERN MATCHING FUNCTIONS (code1)
# =============================================================================

def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    x_bins = np.clip((norm[:, 0] * n_bins).astype(int), 0, n_bins - 1)
    y_bins = np.clip((norm[:, 1] * n_bins).astype(int), 0, n_bins - 1)
    flat_idx = y_bins * n_bins + x_bins
    hist = np.bincount(flat_idx, weights=values, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    counts = np.bincount(flat_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    hist_min, hist_max = hist.min(), hist.max()
    if hist_max > hist_min:
        hist = (hist - hist_min) / (hist_max - hist_min)
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
    norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
    centroid = norm.mean(axis=0)
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() + 1e-8
    ring_idx = np.clip((distances / max_dist * n_rings).astype(int), 0, n_rings - 1)
    profile = np.bincount(ring_idx, weights=values, minlength=n_rings)
    counts = np.bincount(ring_idx, minlength=n_rings)
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile, dtype=float))
    prof_min, prof_max = profile.min(), profile.max()
    if prof_max > prof_min:
        profile = (profile - prof_min) / (prof_max - prof_min)
    return profile


def compute_morans_i_vectorized(coords: np.ndarray, values: np.ndarray, indices: np.ndarray) -> float:
    n = len(values)
    mean_val = values.mean()
    deviations = values - mean_val
    denom = np.sum(deviations ** 2)
    if denom == 0:
        return 0.0
    neighbor_deviations = deviations[indices[:, 1:]]
    numer = np.sum(deviations[:, np.newaxis] * neighbor_deviations)
    w_sum = indices.shape[0] * (indices.shape[1] - 1)
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0.0


@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None


def compute_coordinate_based_similarity(sig1: SpatialSignature, sig2: SpatialSignature, grid_res: int = 50) -> dict:
    """Compute coordinate-based similarity between two signatures in the same coordinate space."""
    coords = sig1.coordinates  # Same coordinates for both (same sample)
    
    # Direct value correlation (no grid interpolation needed - same coordinates)
    mask = (sig1.raw_values > 0) | (sig2.raw_values > 0)
    value_corr = 0
    if mask.sum() > 10:
        r, _ = pearsonr(sig1.raw_values, sig2.raw_values)
        value_corr = r if not np.isnan(r) else 0
    
    # Importance correlation
    imp_corr = 0
    if len(sig1.node_importance) > 10:
        r, _ = pearsonr(sig1.node_importance, sig2.node_importance)
        imp_corr = r if not np.isnan(r) else 0
    
    # Importance IoU
    imp1 = sig1.node_importance / (sig1.node_importance.max() + 1e-8)
    imp2 = sig2.node_importance / (sig2.node_importance.max() + 1e-8)
    importance_iou = np.minimum(imp1, imp2).sum() / (np.maximum(imp1, imp2).sum() + 1e-8)
    
    # Intensity ratio consistency (key for isotope detection)
    intensity_ratio_consistency = compute_intensity_ratio_consistency(sig1.raw_values, sig2.raw_values)
    
    # Peak colocalization
    peak_colocalization = compute_peak_colocalization(sig1.raw_values, sig2.raw_values)
    
    return {
        'value_correlation': value_corr, 
        'importance_iou': importance_iou, 
        'importance_correlation': imp_corr,
        'intensity_ratio_consistency': intensity_ratio_consistency,
        'peak_colocalization': peak_colocalization
    }


def compute_intensity_ratio_consistency(values1: np.ndarray, values2: np.ndarray, min_intensity_pct: float = 10) -> float:
    """
    Compute how consistent the intensity ratio is across pixels.
    For true isotopes, M+1/M+0 ratio should be relatively constant.
    Returns a score from 0 to 1, where 1 = perfectly consistent ratio.
    """
    thresh1 = np.percentile(values1, min_intensity_pct)
    thresh2 = np.percentile(values2, min_intensity_pct)
    
    mask = (values1 > thresh1) & (values2 > thresh2)
    
    if mask.sum() < 10:
        return 0.0
    
    v1 = values1[mask]
    v2 = values2[mask]
    
    ratios = np.minimum(v1, v2) / (np.maximum(v1, v2) + 1e-8)
    
    ratio_mean = ratios.mean()
    ratio_std = ratios.std()
    
    if ratio_mean > 0:
        cv = ratio_std / ratio_mean
        consistency = 1 / (1 + cv)
    else:
        consistency = 0.0
    
    return consistency


def compute_peak_colocalization(values1: np.ndarray, values2: np.ndarray, top_pct: float = 20) -> float:
    """
    Compute overlap of high-intensity regions between two m/z features.
    Returns IoU of top percentile pixels.
    """
    thresh1 = np.percentile(values1, 100 - top_pct)
    thresh2 = np.percentile(values2, 100 - top_pct)
    
    peaks1 = values1 >= thresh1
    peaks2 = values2 >= thresh2
    
    intersection = (peaks1 & peaks2).sum()
    union = (peaks1 | peaks2).sum()
    
    if union > 0:
        return intersection / union
    return 0.0


def compute_descriptor_similarity(sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
    def safe_pearsonr(a, b):
        if a.std() > 0 and b.std() > 0:
            r, _ = pearsonr(a, b)
            return r if not np.isnan(r) else 0
        return 0
    
    spatial_hist_corr = safe_pearsonr(sig1.spatial_histogram.flatten(), sig2.spatial_histogram.flatten())
    radial_corr = safe_pearsonr(sig1.radial_profile, sig2.radial_profile)
    morans_sim = 1 - abs(sig1.morans_i - sig2.morans_i)
    
    return {'spatial_hist_corr': spatial_hist_corr,
            'radial_corr': radial_corr, 
            'morans_similarity': morans_sim}


def compute_combined_score(coord_sim: dict, desc_sim: dict) -> float:
    """
    Compute combined score optimized for isotope detection.
    """
    coord_score = (
        0.0376 * max(coord_sim['intensity_ratio_consistency'], 0) +
        0.0325 * max(coord_sim['value_correlation'], 0) +
        0.1025 * max(coord_sim['peak_colocalization'], 0) +
        0.1490 * coord_sim['importance_iou'] +
        0.1649 * max(coord_sim['importance_correlation'], 0)
    )
    
    desc_score = (
        0.2072 * max(desc_sim['spatial_hist_corr'], 0) +
        0.1572 * max(coord_sim.get('value_iou', 0.5), 0) +
        0.1491 * max(desc_sim['morans_similarity'], 0)
    )
    
    return coord_score + desc_score


def classify_mass_difference(mz_diff: float, tolerance: float = MASS_DIFF_TOLERANCE) -> str:
    """Classify the mass difference into isotope/adduct type."""
    for label, expected_diff in MASS_DIFFS.items():
        if abs(mz_diff - expected_diff) <= tolerance:
            return label
    return 'None'


class MzIsotopeMatcher:
    def __init__(self, output_dir: str = './mz_isotope_matching_results', n_jobs: int = -1):
        self.output_dir = output_dir
        self.n_jobs = n_jobs
        os.makedirs(output_dir, exist_ok=True)
        self.msi_data = {}
        self._nn_cache = {}

    def load_all_data(self):
        print(f"Loading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
            else:
                print(f"  {sample_id}: NOT FOUND at {path}")

    def _get_nn_indices(self, coords: np.ndarray, k: int, cache_key: str) -> np.ndarray:
        full_key = f"{cache_key}_{k}"
        if full_key not in self._nn_cache:
            coord_min, coord_max = coords.min(axis=0), coords.max(axis=0)
            norm = (coords - coord_min) / (coord_max - coord_min + 1e-8)
            k_actual = min(k + 1, len(coords))
            nn = NearestNeighbors(n_neighbors=k_actual)
            nn.fit(norm)
            _, indices = nn.kneighbors(norm)
            self._nn_cache[full_key] = indices
        return self._nn_cache[full_key]

    def compute_bio_importance(self, coords: np.ndarray, values: np.ndarray, k: int, nn_indices: np.ndarray) -> np.ndarray:
        neighbor_vals = values[nn_indices[:, 1:]]
        local_var = np.var(neighbor_vals, axis=1)
        lv_min, lv_max = local_var.min(), local_var.max()
        if lv_max > lv_min:
            local_var = (local_var - lv_min) / (lv_max - lv_min)
        else:
            local_var = np.zeros_like(local_var)
        v_min, v_max = values.min(), values.max()
        if v_max > v_min:
            val_norm = (values - v_min) / (v_max - v_min)
        else:
            val_norm = np.zeros_like(values)
        return 0.5 * local_var + 0.5 * val_norm

    def extract_signature(self, coords: np.ndarray, values: np.ndarray, sample_id: str,
                          feature_name: str, n_neighbors: int, nn_indices: np.ndarray) -> SpatialSignature:
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors, nn_indices)
        return SpatialSignature(
            sample_id=sample_id, feature_name=feature_name, feature_type='mz',
            node_importance=bio_imp,
            spatial_histogram=compute_spatial_histogram(coords, values),
            radial_profile=compute_radial_profile(coords, values),
            morans_i=compute_morans_i_vectorized(coords, values, nn_indices),
            coordinates=coords, raw_values=values)

    def compute_pair_similarity(self, sig1: SpatialSignature, sig2: SpatialSignature) -> dict:
        """Compute all similarity metrics between two m/z signatures."""
        coord_sim = compute_coordinate_based_similarity(sig1, sig2)
        desc_sim = compute_descriptor_similarity(sig1, sig2)
        combined = compute_combined_score(coord_sim, desc_sim)
        
        try:
            mz1 = float(sig1.feature_name)
            mz2 = float(sig2.feature_name)
            mz_diff = abs(mz2 - mz1)
            mass_diff_type = classify_mass_difference(mz_diff)
        except ValueError:
            mz_diff = np.nan
            mass_diff_type = 'None'
        
        return {
            'mz_1': sig1.feature_name,
            'mz_2': sig2.feature_name,
            'mz_difference': mz_diff,
            'mass_diff_type': mass_diff_type,
            'sample_id': sig1.sample_id,
            **coord_sim,
            **desc_sim,
            'combined_score': combined,
            'mz_1_morans_i': sig1.morans_i,
            'mz_2_morans_i': sig2.morans_i
        }

    def run_analysis(self):
        print("\n" + "=" * 70)
        print("m/z-to-m/z MATCHING FOR ISOTOPE DETECTION")
        print(f"MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print(f"Mass differences: {list(MASS_DIFFS.keys())}")
        print(f"Tolerance: ±{MASS_DIFF_TOLERANCE} Da")
        print("=" * 70)

        all_results = []

        for sample_id in tqdm(MSI_SAMPLE_IDS, desc="Samples", unit="sample"):
            if sample_id not in self.msi_data:
                print(f"\n{sample_id}: NOT LOADED, skipping")
                continue

            print(f"\n{'=' * 50}")
            print(f"Sample: {sample_id}")
            print(f"{'=' * 50}")

            msi_adata = self.msi_data[sample_id]
            msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
            msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
            mz_names = list(msi_adata.var_names)
            n_mz = len(mz_names)

            print(f"  {n_mz} m/z features, {len(msi_coords)} pixels")

            # Pre-compute nearest neighbors
            nn_indices = self._get_nn_indices(msi_coords, 8, f"msi_{sample_id}")

            # Compare only pairs with relevant mass differences
            print(f"  Finding candidate isotope/adduct pairs...")
            
            # Parse m/z values
            mz_values = []
            for name in mz_names:
                try:
                    mz_values.append(float(name))
                except ValueError:
                    mz_values.append(np.nan)
            mz_values = np.array(mz_values)
            
            # Find pairs with mass differences matching isotope/adduct patterns
            candidate_pairs = []
            for i in range(n_mz):
                if np.isnan(mz_values[i]):
                    continue
                for j in range(i + 1, n_mz):
                    if np.isnan(mz_values[j]):
                        continue
                    mz_diff = abs(mz_values[j] - mz_values[i])
                    for diff_type, expected_diff in MASS_DIFFS.items():
                        if abs(mz_diff - expected_diff) <= MASS_DIFF_TOLERANCE:
                            candidate_pairs.append((i, j, diff_type))
                            break
            
            n_candidates = len(candidate_pairs)
            print(f"  {n_candidates} candidate pairs (from {n_mz * (n_mz - 1) // 2} total possible)")
            
            if n_candidates == 0:
                print(f"  No candidate pairs found, skipping sample")
                continue
            
            # Extract signatures only for m/z features involved in candidate pairs
            involved_indices = set()
            for i, j, _ in candidate_pairs:
                involved_indices.add(i)
                involved_indices.add(j)
            involved_indices = sorted(involved_indices)
            
            print(f"  Extracting signatures for {len(involved_indices)} involved m/z features...")
            def extract_single_mz(i):
                return i, self.extract_signature(msi_coords, msi_data[:, i], sample_id, mz_names[i], 8, nn_indices)

            sig_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(extract_single_mz)(i) for i in tqdm(involved_indices, desc="  Signatures", unit="mz"))
            
            signatures = {i: sig for i, sig in sig_results}
            print(f"  {len(signatures)} signatures extracted")

            # Compute self-scores for each m/z (maximum possible score)
            print(f"  Computing self-scores...")
            self_scores = {}
            for i in involved_indices:
                self_score_result = self.compute_pair_similarity(signatures[i], signatures[i])
                self_scores[i] = self_score_result['combined_score']
            
            # Compute similarities for candidate pairs
            print(f"  Computing pairwise similarities...")
            
            def compute_pair(i, j, diff_type):
                result = self.compute_pair_similarity(signatures[i], signatures[j])
                result['mass_diff_type'] = diff_type
                result['mz_1_self_score'] = self_scores[i]
                result['mz_2_self_score'] = self_scores[j]
                max_self_score = max(self_scores[i], self_scores[j])
                result['score_percentage'] = (result['combined_score'] / max_self_score * 100) if max_self_score > 0 else 0
                return result

            pair_results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
                delayed(compute_pair)(i, j, diff_type) for i, j, diff_type in tqdm(candidate_pairs, desc="  Pairs", unit="pair"))

            sample_results = pd.DataFrame(pair_results)
            all_results.append(sample_results)

            print(f"  Sample complete: {len(sample_results)} pairs scored")

        # Save all results
        if all_results:
            print("\n" + "=" * 70)
            print("SAVING RESULTS")
            print("=" * 70)

            full_results = pd.concat(all_results, ignore_index=True)
            full_results = full_results.sort_values(['sample_id', 'mass_diff_type', 'combined_score'], ascending=[True, True, False])
            full_path = os.path.join(self.output_dir, 'mz_to_mz_isotope_candidates.csv')
            full_results.to_csv(full_path, index=False)
            print(f"  All candidate pairs: {full_path} ({len(full_results)} rows)")

            summary = []
            for sample_id in MSI_SAMPLE_IDS:
                if sample_id in self.msi_data:
                    sample_data = full_results[full_results['sample_id'] == sample_id]
                    summary.append({
                        'sample_id': sample_id,
                        'n_mz_features': len(self.msi_data[sample_id].var_names),
                        'n_pairs': len(sample_data),
                        'mean_combined_score': sample_data['combined_score'].mean(),
                        'max_combined_score': sample_data['combined_score'].max(),
                        'min_combined_score': sample_data['combined_score'].min(),
                        'std_combined_score': sample_data['combined_score'].std()
                    })
            
            summary_df = pd.DataFrame(summary)
            summary_path = os.path.join(self.output_dir, 'mz_matching_summary.csv')
            summary_df.to_csv(summary_path, index=False)
            print(f"  Summary: {summary_path}")

            return full_results
        
        return None


# =============================================================================
# STAGE 2: ISOTOPE IDENTIFICATION FUNCTIONS (code2)
# =============================================================================

def identify_isotopes(df, min_animals=12, min_score=60):
    """
    Identify m/z values as isotopes if they appear in at least min_animals
    out of 16 animals and have a score_percentage greater than min_score.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing isotope candidate data (from Stage 1)
    min_animals : int
        Minimum number of animals required (default: 12)
    min_score : float
        Minimum score_percentage required (default: 60)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame containing identified isotopes with summary statistics
    """
    
    print(f"Total rows in dataset: {len(df)}")
    print(f"Unique samples: {df['sample_id'].nunique()}")
    print(f"Mass difference types: {df['mass_diff_type'].unique()}")
    
    # Ensure numeric types for m/z and score columns
    df['mz_1'] = pd.to_numeric(df['mz_1'], errors='coerce')
    df['mz_2'] = pd.to_numeric(df['mz_2'], errors='coerce')
    df['score_percentage'] = pd.to_numeric(df['score_percentage'], errors='coerce')
    
    # Round m/z values to 3 decimal places
    df['mz_1_rounded'] = df['mz_1']#.round(2)
    df['mz_2_rounded'] = df['mz_2']#.round(2)
    
    print(f"\nOriginal unique mz_1 values: {df['mz_1'].nunique()}")
    print(f"Rounded unique mz_1 values: {df['mz_1_rounded'].nunique()}")
    print(f"Original unique mz_2 values: {df['mz_2'].nunique()}")
    print(f"Rounded unique mz_2 values: {df['mz_2_rounded'].nunique()}")
    
    # Filter for rows with score_percentage > min_score
    df_filtered = df[df['score_percentage'] > min_score].copy()
    print(f"\nRows with score_percentage > {min_score}: {len(df_filtered)}")
    
    # Create a unique identifier for each isotope pair
    df_filtered['pair_id'] = df_filtered.apply(
        lambda row: f"{row['mz_1_rounded']:.3f}_{row['mz_2_rounded']:.3f}_{row['mass_diff_type']}", 
        axis=1
    )
    
    # Count animals per isotope pair
    isotope_stats = []
    
    for pair_id, group in df_filtered.groupby('pair_id'):
        n_animals = group['sample_id'].nunique()
        
        # Only include pairs that appear in at least min_animals
        if n_animals >= min_animals:
            stats = {
                'mz_1': group['mz_1_rounded'].iloc[0],
                'mz_2': group['mz_2_rounded'].iloc[0],
                'mz_difference': group['mz_difference'].mean(),
                'mass_diff_type': group['mass_diff_type'].iloc[0],
                'n_animals': n_animals,
                'mean_score_percentage': group['score_percentage'].mean(),
                'median_score_percentage': group['score_percentage'].median(),
                'min_score_percentage': group['score_percentage'].min(),
                'max_score_percentage': group['score_percentage'].max(),
                'std_score_percentage': group['score_percentage'].std(),
                'animals': ','.join(sorted(group['sample_id'].unique()))
            }
            isotope_stats.append(stats)
    
    # Create results DataFrame
    results_df = pd.DataFrame(isotope_stats)
    
    if len(results_df) > 0:
        results_df = results_df.sort_values(
            ['n_animals', 'mean_score_percentage'], 
            ascending=[False, False]
        ).reset_index(drop=True)
    
    return results_df, df_filtered

def save_results(results_df, output_file='identified_isotopes.csv'):
    """Save identified isotopes to CSV file"""
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")

def print_summary(results_df, min_animals=12, min_score=60):
    """Print summary statistics"""
    print("\n" + "="*80)
    print(f"ISOTOPE IDENTIFICATION SUMMARY")
    print(f"Criteria: ≥{min_animals} animals AND score_percentage > {min_score}")
    print("="*80)
    
    if len(results_df) == 0:
        print("\nNo isotopes found meeting the criteria.")
        return
    
    print(f"\nTotal isotope pairs identified: {len(results_df)}")
    print(f"\nBreakdown by mass difference type:")
    for mass_type, count in results_df['mass_diff_type'].value_counts().items():
        print(f"  {mass_type}: {count}")
    
    print(f"\nBreakdown by number of animals:")
    for n_animals in sorted(results_df['n_animals'].unique(), reverse=True):
        count = (results_df['n_animals'] == n_animals).sum()
        print(f"  {n_animals} animals: {count} pairs")
    
    print(f"\nScore statistics for identified isotopes:")
    print(f"  Mean score_percentage: {results_df['mean_score_percentage'].mean():.2f}")
    print(f"  Median score_percentage: {results_df['median_score_percentage'].median():.2f}")
    print(f"  Range: {results_df['min_score_percentage'].min():.2f} - {results_df['max_score_percentage'].max():.2f}")
    
    print(f"\nTop 10 isotope pairs by mean score_percentage:")
    print("-"*80)
    top_10 = results_df.head(10)
    for idx, row in top_10.iterrows():
        mz1 = float(row['mz_1'])
        mz2 = float(row['mz_2'])
        print(f"{idx+1}. m/z {mz1:.3f} → {mz2:.3f} ({row['mass_diff_type']})")
        print(f"   Animals: {row['n_animals']}/16 | Mean score: {row['mean_score_percentage']:.2f}%")
        print()


# =============================================================================
# STAGE 3: PARENT-CHILDREN HIERARCHY (code3)
# =============================================================================

def build_strict_hierarchy(path):
    """
    Build parent-children hierarchy from identified isotopes CSV.
    
    Parameters:
    -----------
    path : str
        Path to the identified_isotopes.csv file (output of Stage 2)
    
    Returns:
    --------
    pd.DataFrame or None
        DataFrame with Parent_MZ and Child columns, or None if no matches found
    """
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Error: Could not find file at {path}")
        return None

    # Get all unique m/z values and round them for consistency
    all_mzs = sorted(pd.concat([df['mz_1'], df['mz_2']]).unique())
    all_mzs = np.array([round(x, HIERARCHY_PRECISION) for x in all_mzs])
    
    rows = []
    assigned_as_child = set()

    for parent in all_mzs:
        # Skip if this MZ has already been claimed as a child by a smaller parent
        if parent in assigned_as_child:
            continue
            
        row_dict = {'Parent_MZ': parent}
        found_any_child = False
        
        # Check each specific pattern
        for i, (label, diff) in enumerate(MASS_DIFFS.items()):
            target_mz = parent + diff
            
            # Find all candidates within tolerance
            distances = np.abs(all_mzs - target_mz)
            candidates_mask = distances <= HIERARCHY_TOLERANCE
            
            if np.any(candidates_mask):
                # Get the candidate indices
                indices = np.where(candidates_mask)[0]
                # Choose the index with the minimum distance (closest match)
                best_match_idx = indices[np.argmin(distances[indices])]
                best_match = all_mzs[best_match_idx]
                
                # Assign to the specific child slot
                row_dict[f'Child_{i+1}'] = best_match
                assigned_as_child.add(best_match)
                found_any_child = True
            else:
                row_dict[f'Child_{i+1}'] = np.nan
        
        # Only keep the row if at least one child (isotope/adduct) was found
        if found_any_child:
            rows.append(row_dict)

    if not rows:
        print("No matches found based on the provided MASS_DIFFS.")
        return None
        
    final_df = pd.DataFrame(rows)
    
    # Ensure correct column ordering
    cols = ['Parent_MZ'] + [f'Child_{i+1}' for i in range(len(MASS_DIFFS))]
    return final_df[cols]


# =============================================================================
# MAIN: CONNECT STAGE 1 → STAGE 2 → STAGE 3
# =============================================================================

def main():
    print("=" * 70)
    print("m/z-to-m/z Isotope Pattern Matching, Identification & Hierarchy Pipeline")
    print(f"MSI: {MSI_PIXEL_SIZE}μm")
    print("=" * 70)
    
    # --- STAGE 1: Spatial pattern matching ---
    matcher = MzIsotopeMatcher(output_dir=OUTPUT_DIR, n_jobs=-1)
    matcher.load_all_data()
    matching_results = matcher.run_analysis()
    
    if matching_results is None:
        print("\nNo matching results produced. Exiting.")
        return matcher, None, None, None
    
    # --- STAGE 2: Isotope identification (uses Stage 1 DataFrame directly) ---
    print("\n" + "=" * 70)
    print("STAGE 2: ISOTOPE IDENTIFICATION")
    print("=" * 70)
    print(f"\nStarting isotope identification analysis...")
    
    results_df, filtered_df = identify_isotopes(
        matching_results,
        min_animals=MIN_ANIMALS,
        min_score=MIN_SCORE
    )
    
    # Print summary
    print_summary(results_df, MIN_ANIMALS, MIN_SCORE)
    
    # Save Stage 2 results
    isotope_csv_path = None
    if len(results_df) > 0:
        isotope_csv_path = os.path.join(OUTPUT_DIR, 'identified_isotopes.csv')
        save_results(results_df, isotope_csv_path)
        
        detailed_output = isotope_csv_path.replace('.csv', '_detailed.csv')
        filtered_df.to_csv(detailed_output, index=False)
        print(f"Detailed data saved to {detailed_output}")
    
    # --- STAGE 3: Parent-children hierarchy (uses Stage 2 CSV output) ---
    print("\n" + "=" * 70)
    print("STAGE 3: PARENT-CHILDREN HIERARCHY")
    print("=" * 70)
    
    hierarchy_df = None
    if isotope_csv_path is not None and os.path.exists(isotope_csv_path):
        print(f"\nBuilding hierarchy from: {isotope_csv_path}")
        hierarchy_df = build_strict_hierarchy(isotope_csv_path)
        
        if hierarchy_df is not None:
            print(f"\nSuccessfully grouped {len(hierarchy_df)} parent-centered families.")
            print(f"Logic: Closest match within {HIERARCHY_TOLERANCE} Da selected for each pattern.")
            print(hierarchy_df.head().to_string(float_format=f"%.{HIERARCHY_PRECISION}f"))
            
            hierarchy_output_path = os.path.join(OUTPUT_DIR, 'parent_children_hierarchy.csv')
            hierarchy_df.to_csv(hierarchy_output_path, index=False, float_format=f"%.{HIERARCHY_PRECISION}f")
            print(f"\nFinal table saved to: {hierarchy_output_path}")
        else:
            print("\nNo hierarchy could be built from the identified isotopes.")
    else:
        print("\nNo identified isotopes CSV available. Skipping hierarchy construction.")
    
    print("\n" + "=" * 70)
    print("COMPLETE!")
    print("=" * 70)
    
    return matcher, matching_results, results_df, hierarchy_df


if __name__ == "__main__":
    matcher, matching_results, isotope_results, hierarchy_results = main()


m/z-to-m/z Isotope Pattern Matching, Identification & Hierarchy Pipeline
MSI: 60μm
Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

m/z-to-m/z MATCHING FOR ISOTOPE DETECTION
MSI: 60μm (Cartesian)
Mass differences: ['Isotope (M+1)', 'Isotope (M+2)', 'Adduct (NH4)', 'Adduct (Na)', 'Adduct (K)']
Tolerance: ±0.01 Da


Samples:   0%|          | 0/16 [00:00<?, ?sample/s]


Sample: YC_1
  528 m/z features, 6688 pixels
  Finding candidate isotope/adduct pairs...
  773 candidate pairs (from 139128 total possible)
  Extracting signatures for 492 involved m/z features...


  Signatures: 100%|██████████| 492/492 [00:00<00:00, 836.12mz/s] 


  492 signatures extracted
  Computing self-scores...
  Computing pairwise similarities...


...

  Signatures: 100%|██████████| 492/492 [00:00<00:00, 1061.14mz/s]


  492 signatures extracted
  Computing self-scores...
  Computing pairwise similarities...


Samples: 100%|██████████| 16/16 [01:52<00:00,  7.03s/sample]

  Sample complete: 773 pairs scored

SAVING RESULTS


  All candidate pairs: ./mz_isotope_results_1/mz_to_mz_isotope_candidates.csv (12368 rows)
  Summary: ./mz_isotope_results_1/mz_matching_summary.csv

STAGE 2: ISOTOPE IDENTIFICATION

Starting isotope identification analysis...
Total rows in dataset: 12368
Unique samples: 16
Mass difference types: ['Adduct (K)' 'Adduct (NH4)' 'Adduct (Na)' 'Isotope (M+1)' 'Isotope (M+2)']

Original unique mz_1 values: 396
Rounded unique mz_1 values: 396
Original unique mz_2 values: 413
Rounded unique mz_2 values: 413

Rows with score_percentage > 60: 9067

ISOTOPE IDENTIFICATION SUMMARY
Criteria: ≥12 animals AND score_percentage > 60

Total isotope pairs identified: 520

Breakdown by mass difference type:
  Isotope (M+1): 279
  Isotope (M+2): 159
  Adduct (Na): 42
  Adduct (NH4): 21
  Adduct (K): 19

Breakdown by number of animals:
  16 animals: 353 pairs
  15 animals: 55 pairs
  14 animals: 45 pairs
  13 animals: 32 pairs
  12 animals: 35 pairs

Score statistics for identified isotopes:
  Mean score_pe